In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install descartes

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/WIRELESSCOM/wirelesscode')
from utils.fileGen import fileGen

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/WIRELESSCOM/wirelesscode')
from utils.fileGen import fileGen

FEATURE_PATH = "/content/drive/MyDrive/WIRELESSCOM/wirelesscode/raw_data/feature_matrix.csv"
OUTPUT_PATH = "/content/drive/MyDrive/WIRELESSCOM/wirelesscode/raw_data/output_matrix.csv"
IMAGE_PATH = "/content/drive/MyDrive/WIRELESSCOM/wirelesscode/raw_data/mapbox_api/"
tofile = True

file_path = '/content/drive/MyDrive/WIRELESSCOM/wirelesscode/utils/drive_test_route_journal.py'

# Read existing file content
with open(file_path, 'r') as f:
    lines = f.readlines()

new_draw_boundary_boxes_content_str = """    def draw_boundary_boxes(self):
        ax = plt.gca()
        for box_gs in self.boundary_box:
            if not box_gs.empty:
                geom = box_gs.iloc[0] # Use geom as it could be MultiPolygon
                polygons_to_draw = []
                if geom.geom_type == "Polygon":
                    polygons_to_draw.append(geom)
                elif geom.geom_type == "MultiPolygon":
                    polygons_to_draw.extend(list(geom.geoms))
                for polygon in polygons_to_draw:
                    if polygon.is_valid and not polygon.is_empty:
                        # Check exterior ring
                        exterior_ok = (hasattr(polygon, "exterior") and
                                       polygon.exterior is not None and
                                       len(polygon.exterior.coords) >= 3)
                        # Check all interior rings
                        interiors_ok = True
                        if hasattr(polygon, "interiors") and polygon.interiors is not None:
                            for interior_ring in polygon.interiors:
                                if not (hasattr(interior_ring, "coords") and
                                        interior_ring.coords is not None and
                                        len(interior_ring.coords) >= 3):
                                    interiors_ok = False
                                    break
                        if exterior_ok and interiors_ok:
                            try:
                                ax.add_patch(PolygonPatch(polygon, alpha=0.2))
                            except IndexError as e:
                                print(f"Skipping problematic polygon: {polygon} due to {e}")
"""

# Find the start and end of the draw_boundary_boxes function
start_idx = -1
end_idx = -1
for i, line in enumerate(lines):
    if 'def draw_boundary_boxes(self):' in line:
        start_idx = i
        # Determine indentation level of the function definition
        indent_level = len(line) - len(line.lstrip())
        # Find the end of the function body
        for j in range(i + 1, len(lines)):
            stripped_line = lines[j].lstrip()
            # If line is empty or starts with less indentation, or starts a new def/class
            if (not stripped_line) or \
               (len(lines[j]) - len(stripped_line) <= indent_level and stripped_line not in ['\n']) or \
               stripped_line.startswith('def ') or \
               stripped_line.startswith('class '):
                end_idx = j
                break
        if end_idx == -1: # If function goes to end of file
            end_idx = len(lines)
        break

if start_idx != -1:
    # Replace the function content
    modified_lines_content = (
        lines[:start_idx] +
        [new_draw_boundary_boxes_content_str] +
        lines[end_idx:]
    )
else:
    # Function not found, append original lines and print a warning
    modified_lines_content = lines
    print("Warning: 'def draw_boundary_boxes(self):' not found in the file.")

# Apply .ix to .iloc replacements everywhere
final_modified_lines = []
for line in modified_lines_content:
    if 'boundary_box.ix[0]' in line:
        final_modified_lines.append(line.replace('boundary_box.ix[0]', 'boundary_box.iloc[0]'))
    else:
        final_modified_lines.append(line)

# Overwrite the file with modified content
with open(file_path, 'w') as f:
    f.writelines(final_modified_lines)

print(f"Updated {file_path} with robust function replacement for draw_boundary_boxes and .ix to .iloc fix.")

# Add module reloading to ensure updated code is used
import importlib
import sys

# Reload drive_test_route_journal if it's already loaded
if 'utils.drive_test_route_journal' in sys.modules:
    importlib.reload(sys.modules['utils.drive_test_route_journal'])
    print("Reloaded utils.drive_test_route_journal.")

# Reload fileGen itself to ensure it picks up the reloaded get_training_test_data
if 'utils.fileGen' in sys.modules:
    importlib.reload(sys.modules['utils.fileGen'])
    # Re-import the class from the reloaded module
    from utils.fileGen import fileGen
    print("Reloaded utils.fileGen and re-imported fileGen class.")
else:
    # If not loaded, just ensure it's imported (less likely given previous executions)
    from utils.fileGen import fileGen

# Re-run the problematic code after applying the fix and reloading modules
if tofile:
    file_generator = fileGen(FEATURE_PATH, OUTPUT_PATH)
    file_generator.generate_files()